# Task 3, Step 1: Implementing and Validating Exact Speculative Sampling

Our sampler (`src/sampling.py`, written from scratch): draft proposals, one target verification pass, stochastic
acceptance min(1, p/q), positive-residual resampling, the full-acceptance bonus token, and cache rollback /
advancement (each cache always holds at most seq[:-1]; rejected proposals are cropped; the last accepted proposal is
fed to the draft at the start of the next round).

1. **Toy multi-step Markov test:** 3-symbol target and draft chains whose joint law over length-3 sequences is
   computed exactly; 100,000 fixed-seed sequences per sampler; Pearson chi-square on all 27 joint categories
   (every expected count >= 10). The correct sampler must not be rejected at p < 0.01; a negative control that samples
   from p instead of the positive residual must be rejected.
2. **Mechanics:** variable proposal lengths, final-token truncation, one-token rounds, fully accepted rounds, EOS.
3. **Real models:** tokenizer / chat-template agreement, split-prefill and cache-position equivalence, and an
   identical-draft check (draft = target must accept every proposal).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from shared.src.io import save_json
from shared.src.plotting import save_figure
from task3_speculative.src import sampling as S

TARGET = S.MarkovLM([0.5, 0.3, 0.2], [[0.6, 0.3, 0.1], [0.2, 0.5, 0.3], [0.3, 0.3, 0.4]])
DRAFT = S.MarkovLM([0.2, 0.3, 0.5], [[0.2, 0.2, 0.6], [0.5, 0.2, 0.3], [0.1, 0.6, 0.3]])
N = 100_000
joint = TARGET.joint(3)
OUT = ctx.results / "sampler"
OUT.mkdir(exist_ok=True)

In [ ]:
tests, counts_all = [], {}
for label, gamma, residual, clip in [("exact gamma=1", 1, "exact", True), ("exact gamma=2", 2, "exact", True),
                                     ("exact gamma=4 (no clip, truncate)", 4, "exact", False),
                                     ("NEGATIVE: residual replaced by p, gamma=2", 2, "p", True)]:
    g = torch.Generator().manual_seed(6304)
    counts, acc = {}, dict(rounds=0, accepted=0, examined=0, bonus=0)
    for _ in range(N):
        s, st = S.toy_speculative_sample(TARGET, DRAFT, 3, gamma, g, residual, clip)
        counts[s] = counts.get(s, 0) + 1
        for k in acc:
            acc[k] += st[k]
    r = S.chi2_test(counts, joint, N)
    r.update(sampler=label, accept_rate=acc["accepted"] / max(acc["examined"], 1), rejected_at_0_01=r["p_value"] < 0.01)
    tests.append(r)
    counts_all[label] = counts
    print(r, flush=True)
chi = pd.DataFrame(tests).set_index("sampler")
chi.to_csv(OUT / "toy_chi2.csv")
assert not chi.loc["exact gamma=2", "rejected_at_0_01"] and chi.loc["NEGATIVE: residual replaced by p, gamma=2", "rejected_at_0_01"]
chi

In [ ]:
seqs = sorted(joint)
tab = pd.DataFrame({"exact P(x1,x2,x3)": [joint[s] for s in seqs],
                    **{lab: [c.get(s, 0) / N for s in seqs] for lab, c in counts_all.items()}},
                   index=["".join(map(str, s)) for s in seqs])
tab.to_csv(OUT / "toy_joint_frequencies.csv")
ax = tab[["exact P(x1,x2,x3)", "exact gamma=2", "NEGATIVE: residual replaced by p, gamma=2"]].plot(kind="bar", figsize=(15, 4), width=0.8)
ax.set_ylabel("probability / frequency"); ax.set_xlabel("joint sequence"); ax.set_title(f"Joint sequence law, {N:,} samples each")
save_figure(ctx.figures, "toy_joint_sequence_law")
tab.round(4)

## Real models: tokenizer agreement, split prefill, cache positions, identical-draft acceptance

In [ ]:
from task3_speculative.src.data import test_prompts, train_prompts, encode_prompt, eos_ids, check_disjoint
from task3_speculative.src.distill import load_models, check_tokenizers
M = load_models()
tok, target, draft = M["tok"], M["target"], M["draft"]
math_tr, short_tr = train_prompts()
tk = check_tokenizers(test_prompts() + math_tr[:50] + short_tr[:50])
dj = check_disjoint()
print(tk, dj)
assert not tk["mismatches"] and not dj["exact_prompt_overlap"]
assert target.config.vocab_size == draft.config.vocab_size

In [ ]:
from transformers import DynamicCache
checks = []
eos = eos_ids(tok)
for pr in test_prompts()[:3] + test_prompts()[60:62]:
    ids = encode_prompt(tok, pr["prompt"])
    g = torch.Generator(device="cuda").manual_seed(1)
    gen = S.sample_generate(target, ids, 24, g, eos)["tokens"]
    full = ids + gen
    with torch.no_grad():
        ref = target(input_ids=torch.tensor([full], device="cuda")).logits[0].float()
        c = DynamicCache()
        target(input_ids=torch.tensor([full[:len(ids) - 1]], device="cuda"), past_key_values=c, use_cache=True)
        parts, pos = [], len(ids) - 1
        for chunk in (full[pos:pos + 3], full[pos + 3:pos + 4], full[pos + 4:]):   # uneven chunks like spec rounds
            parts.append(target(input_ids=torch.tensor([chunk], device="cuda"), past_key_values=c, use_cache=True).logits[0].float())
        c.crop(len(ids) + 5)                                               # rollback then re-advance
        redo = target(input_ids=torch.tensor([full[len(ids) + 5:]], device="cuda"), past_key_values=c, use_cache=True).logits[0].float()
    split = torch.cat(parts)
    g = torch.Generator(device="cuda").manual_seed(2)
    same = S.speculative_generate(target, target, ids, 4, 32, g, eos)
    checks.append(dict(pid=pr["pid"], split_prefill_max_abs=float((split - ref[len(ids) - 1:]).abs().max()),
                       rollback_readvance_max_abs=float((redo - ref[len(ids) + 5:]).abs().max()),
                       argmax_agree=float((split.argmax(-1) == ref[len(ids) - 1:].argmax(-1)).float().mean()),
                       identical_draft_accept=same["accepted"] / max(same["proposed"], 1), finite=bool(torch.isfinite(ref).all())))
pd.DataFrame(checks)

Acceptance with draft = target should be 1 up to FP16 differences between a 1-token draft step and the multi-token
verification pass (identical weights, different kernel shapes).

In [ ]:
mech = []
pr = test_prompts()[0]
ids = encode_prompt(tok, pr["prompt"])
for gamma, max_new, ign in [(4, 1, False), (4, 7, False), (2, 64, False), (4, 64, True)]:
    g = torch.Generator(device="cuda").manual_seed(3)
    r = S.speculative_generate(target, draft, ids, gamma, max_new, g, eos, ignore_eos=ign, record_rounds=True)
    mech.append(dict(gamma=gamma, max_new=max_new, ignore_eos=ign, n_out=r["n_out"], eos=r["eos"], truncated=r["truncated"],
                     rounds=r["rounds"], proposals_per_round=[x["g"] for x in r["rounds_detail"]][:12],
                     accepted_per_round=[x["n"] for x in r["rounds_detail"]][:12], bonus_rounds=r["bonus"]))
mech_df = pd.DataFrame(mech)
assert mech_df.loc[0, "n_out"] == 1 and mech_df.loc[0, "proposals_per_round"] == [0]
save_json(OUT / "real_model_checks.json", dict(tokenizers=tk, disjoint=dj, cache=checks, mechanics=mech))
mech_df